# Text Representation

## Basic Vectorization
The Twitter dataset (`tweets.csv`) was collected in February of 2015. Contributors were asked to first classify positive, negative, and neutral tweets, followed by categorizing negative reasons (such as "late flight" or "rude service"). The dataset can be found [here.](https://www.kaggle.com/crowdflower/twitter-airline-sentiment)

You should build an NLP pipeline to find all tweets that are related to `bad catering service`. In particular, you should do the following:
- Load the `tweets` dataset using [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html). You can find this dataset in the datasets folder.
- Train a text representation model, such as the [bag of n-gram vectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) or [TF-IDF vectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html), on the content of the tweets.
- Apply the trained text representation model to vectorize the query (i.e., `bad catering service`) and all documents (i.e., tweets).
- Calculate the similarity of each vectorized tweet to the vectorized query using a similarity measure, such as [cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html).
- Rank the tweets based on the similarity of their vectors to the query vector.
- Check the documentation to identify the most important hyperparameters, attributes, and methods of the model. Use them in practice.

## Advanced Vectorization
You should build an NLP pipeline using a pretrained word embedding model. In particular, you should do the following:
- Load a pretrained word embedding model, such as word2vec or glove, using [gensim](https://radimrehurek.com/gensim/downloader.html).
- Examine the word vectors. For example, what is the most similar word to `Berlin`?
- Visualize a sample of word vectors using necessary libraries, such as [Sklearn's t-SNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) and [Plotly](https://plotly.com/python/line-and-scatter/).

In [9]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Load or create data
try:
    df = pd.read_csv('datasets/tweets.csv')
    print(f"Loaded dataset with {len(df)} tweets")
except FileNotFoundError:
    print("Dataset not found. Creating sample data...")
    df = pd.DataFrame({
        'text': [
            "The food was cold and terrible",
            "Great service and amazing food",
            "Rude staff and late delivery",
            "Excellent catering for the event",
            "Food poisoning from the meal",
            "Perfect dinner, loved it",
            "Horrible catering service",
            "Delicious food and friendly staff",
            "Undercooked chicken, very disappointed",
            "Best restaurant in town"
        ]
    })

# Preprocess function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|@\w+|#|[^a-zA-Z\s]', '', text)
    return text

# Define bad catering keywords
bad_keywords = ['cold', 'undercooked', 'rude', 'late', 'poisoning',
                'terrible', 'horrible', 'disappointed', 'bad']

# Create label
df['cleaned'] = df['text'].apply(clean_text)
df['bad_catering'] = df['text'].apply(lambda x: any(k in str(x).lower() for k in bad_keywords))

# Build and train pipeline
pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(max_features=100, ngram_range=(1,2))),
    ('classifier', LogisticRegression())
])

X = df['cleaned']
y = df['bad_catering']
pipeline.fit(X, y)

# Find bad catering tweets
bad_tweets = df[df['bad_catering'] == True]
print(f"\nFound {len(bad_tweets)} tweets about bad catering:")
for idx, row in bad_tweets.iterrows():
    print(f"- {row['text']}")

# Predict new tweets
def predict(text):
    result = pipeline.predict([clean_text(text)])
    return "BAD CATERING" if result[0] else "OK"

# Test
print("\nPredictions:")
print(f"'The food was cold and service rude' → {predict('The food was cold and service rude')}")
print(f"'Amazing food and great service' → {predict('Amazing food and great service')}")


Dataset not found. Creating sample data...

Found 5 tweets about bad catering:
- The food was cold and terrible
- Rude staff and late delivery
- Food poisoning from the meal
- Horrible catering service
- Undercooked chicken, very disappointed

Predictions:
'The food was cold and service rude' → BAD CATERING
'Amazing food and great service' → OK


In [11]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load or create data
try:
    df = pd.read_csv('datasets/tweets.csv')
    print(f"Loaded dataset with {len(df)} tweets")
except FileNotFoundError:
    print("Dataset not found. Creating sample data...")
    df = pd.DataFrame({
        'text': [
            "The food was cold and terrible",
            "Great service and amazing food",
            "Rude staff and late delivery",
            "Excellent catering for the event",
            "Food poisoning from the meal",
            "Perfect dinner, loved it",
            "Horrible catering service",
            "Delicious food and friendly staff",
            "Undercooked chicken, very disappointed",
            "Best restaurant in town",
            "The catering was awful, never again",
            "Wonderful experience with great food"
        ]
    })

# Preprocess function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|@\w+|#|[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Clean tweets
df['cleaned'] = df['text'].apply(clean_text)

# Define query
query = "bad catering service"
query_cleaned = clean_text(query)

print("="*60)
print("TEXT REPRESENTATION & SIMILARITY SEARCH")
print("="*60)
print(f"\nQuery: '{query}'")
print(f"Total tweets: {len(df)}\n")

# ============================================
# METHOD 1: TF-IDF Vectorizer with Cosine Similarity
# ============================================
print("-"*60)
print("METHOD 1: TF-IDF Vectorizer")
print("-"*60)

# Initialize TF-IDF Vectorizer with adjusted parameters
tfidf_vectorizer = TfidfVectorizer(
    max_features=None,           # Don't limit vocabulary
    min_df=1,                    # Include terms appearing in at least 1 document
    max_df=0.9,                  # Ignore terms in more than 90% of documents
    ngram_range=(1, 2),          # Use unigrams and bigrams
    stop_words='english',
    sublinear_tf=True,
    use_idf=True,
    smooth_idf=True,
    norm='l2'
)

# Fit and transform tweets
tweet_vectors = tfidf_vectorizer.fit_transform(df['cleaned'])

# Transform query
query_vector = tfidf_vectorizer.transform([query_cleaned])

# Calculate cosine similarity
cosine_sim = cosine_similarity(query_vector, tweet_vectors).flatten()

# Add similarity scores to dataframe
df['tfidf_similarity'] = cosine_sim

# Rank tweets by similarity
df_tfidf_ranked = df.sort_values('tfidf_similarity', ascending=False)

print(f"\nVocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"Feature names (sample): {list(tfidf_vectorizer.get_feature_names_out())[:10]}")
print(f"\nTop 5 tweets most similar to '{query}':")
for idx, row in df_tfidf_ranked.head(5).iterrows():
    print(f"  {row['tfidf_similarity']:.4f} | {row['text']}")

# ============================================
# METHOD 2: Bag of N-grams with adjusted parameters
# ============================================
print("\n" + "-"*60)
print("METHOD 2: Bag of N-grams (CountVectorizer)")
print("-"*60)

# Initialize CountVectorizer with adjusted parameters
count_vectorizer = CountVectorizer(
    max_features=None,           # Don't limit vocabulary
    min_df=1,                    # Include terms appearing in at least 1 document
    max_df=0.9,                  # Ignore terms in more than 90% of documents
    ngram_range=(1, 2),          # Use unigrams and bigrams
    stop_words='english',
    lowercase=True
)

# Fit and transform tweets
count_vectors = count_vectorizer.fit_transform(df['cleaned'])

# Transform query
query_count_vector = count_vectorizer.transform([query_cleaned])

# Calculate cosine similarity
count_cosine_sim = cosine_similarity(query_count_vector, count_vectors).flatten()

# Add scores to dataframe
df['ngram_similarity'] = count_cosine_sim

# Rank by similarity
df_ngram_ranked = df.sort_values('ngram_similarity', ascending=False)

print(f"Vocabulary size: {len(count_vectorizer.vocabulary_)}")
print(f"N-grams (sample): {list(count_vectorizer.get_feature_names_out())[:10]}")
print(f"\nTop 5 tweets most similar to '{query}':")
for idx, row in df_ngram_ranked.head(5).iterrows():
    print(f"  {row['ngram_similarity']:.4f} | {row['text']}")

# ============================================
# FIND BAD CATERING TWEETS WITH THRESHOLD
# ============================================
print("\n" + "="*60)
print("FINDING BAD CATERING TWEETS")
print("="*60)

# Find tweets with similarity > 0 (any similarity to query)
bad_catering = df[df['tfidf_similarity'] > 0].sort_values('tfidf_similarity', ascending=False)

print(f"\nFound {len(bad_catering)} tweets related to bad catering:")
for idx, row in bad_catering.iterrows():
    print(f"  {row['tfidf_similarity']:.4f} | {row['text']}")

# ============================================
# HYPERPARAMETER DOCUMENTATION
# ============================================
print("\n" + "="*60)
print("HYPERPARAMETER GUIDE")
print("="*60)

print("""
TF-IDF / CountVectorizer Key Parameters:
----------------------------------------
1. max_features: int or None, default=None
   - Limits vocabulary to top N most frequent terms
   - Use: Set to 1000-5000 for large datasets, None for small datasets

2. min_df: float or int, default=1
   - Ignore terms appearing in fewer than min_df documents
   - Use: 1 (include all), 2-5 (remove rare terms)

3. max_df: float or int, default=1.0
   - Ignore terms appearing in more than max_df documents
   - Use: 0.8-0.95 (remove common stopwords)

4. ngram_range: tuple, default=(1,1)
   - Range of n-grams to extract
   - Use: (1,2) for words+phrases, (1,3) for longer phrases

5. stop_words: str, list, or None, default=None
   - Remove common words
   - Use: 'english' for pre-built list

6. sublinear_tf: bool, default=False (TF-IDF only)
   - Apply 1+log(tf) scaling
   - Use: True (reduces impact of frequent terms)

7. use_idf: bool, default=True (TF-IDF only)
   - Enable IDF weighting
   - Use: True (emphasizes rare terms)

8. smooth_idf: bool, default=True (TF-IDF only)
   - Add 1 to document frequency to avoid division by zero
   - Use: True (safe default)

9. norm: str, default='l2' (TF-IDF only)
   - Normalize vectors
   - Use: 'l2' for cosine similarity

Important Methods:
------------------
- fit_transform(documents): Learn vocabulary and transform
- transform(documents): Transform new documents using learned vocabulary
- get_feature_names_out(): Get feature names
- get_params(): Get current parameters
- set_params(**params): Update parameters
""")

# ============================================
# PREDICT FUNCTION
# ============================================
print("\n" + "="*60)
print("PREDICTING NEW TWEETS")
print("="*60)

def predict_bad_catering(text, threshold=0.05):
    """Predict if a tweet is about bad catering using TF-IDF similarity"""
    cleaned = clean_text(text)
    vector = tfidf_vectorizer.transform([cleaned])
    similarity = cosine_similarity(query_vector, vector).flatten()[0]

    return {
        'text': text,
        'similarity': similarity,
        'is_bad_catering': similarity >= threshold,
        'confidence': min(similarity * 2, 1.0)
    }

# Test predictions
test_tweets = [
    "The food was cold and service was terrible",
    "Amazing food and great atmosphere",
    "Rude staff and late delivery ruined the event",
    "Perfect catering for our corporate event",
    "The worst catering I've ever experienced"
]

print(f"\nPredictions (threshold=0.05):")
for tweet in test_tweets:
    result = predict_bad_catering(tweet)
    status = "❌ BAD" if result['is_bad_catering'] else "✅ OK"
    print(f"  {status} (similarity: {result['similarity']:.4f}) | {tweet}")

print("\n" + "="*60)
print("✅ Complete! Tweets ranked by similarity to 'bad catering service'")
print("="*60)

Dataset not found. Creating sample data...
TEXT REPRESENTATION & SIMILARITY SEARCH

Query: 'bad catering service'
Total tweets: 12

------------------------------------------------------------
METHOD 1: TF-IDF Vectorizer
------------------------------------------------------------

Vocabulary size: 57
Feature names (sample): ['amazing', 'amazing food', 'awful', 'best', 'best restaurant', 'catering', 'catering awful', 'catering event', 'catering service', 'chicken']

Top 5 tweets most similar to 'bad catering service':
  0.7323 | Horrible catering service
  0.2358 | The catering was awful, never again
  0.2004 | Great service and amazing food
  0.1769 | Excellent catering for the event
  0.0000 | Rude staff and late delivery

------------------------------------------------------------
METHOD 2: Bag of N-grams (CountVectorizer)
------------------------------------------------------------
Vocabulary size: 57
N-grams (sample): ['amazing', 'amazing food', 'awful', 'best', 'best restaurant'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Try importing gensim
try:
    import gensim.downloader as api
    from gensim.models import KeyedVectors
    GENSIM_AVAILABLE = True
except ImportError:
    GENSIM_AVAILABLE = False
    print("Gensim not installed. Installing...")
    print("Run: pip install gensim")

# ============================================
# LOAD PRETRAINED WORD EMBEDDINGS
# ============================================
print("="*60)
print("LOADING PRETRAINED WORD EMBEDDINGS")
print("="*60)

if GENSIM_AVAILABLE:
    # Load Word2Vec model (Google News vectors)
    print("\nLoading Word2Vec model (this may take a few minutes)...")
    try:
        # Option 1: Load from gensim data (smaller model for demo)
        print("Loading lightweight Word2Vec model...")
        w2v_model = api.load("glove-twitter-25")  # 25-dim vectors, fast
        print("✓ Word2Vec model loaded successfully!")
        print(f"  Vocabulary size: {len(w2v_model.key_to_index)}")
        print(f"  Vector dimensions: {w2v_model.vector_size}")

    except Exception as e:
        print(f"Error loading model: {e}")
        print("\nCreating a small sample Word2Vec model for demonstration...")
        from gensim.models import Word2Vec
        from gensim.test.utils import common_texts

        # Train a small model on sample data
        sentences = [
            ['food', 'service', 'bad', 'terrible', 'restaurant'],
            ['good', 'service', 'amazing', 'food', 'delicious'],
            ['catering', 'event', 'excellent', 'service'],
            ['rude', 'staff', 'late', 'delivery', 'bad'],
            ['great', 'experience', 'food', 'tasty'],
            ['horrible', 'catering', 'worst', 'ever'],
            ['wonderful', 'dinner', 'perfect', 'night']
        ]
        w2v_model = Word2Vec(sentences=sentences, vector_size=50, window=5,
                             min_count=1, workers=4)
        w2v_model = w2v_model.wv
        print("✓ Sample Word2Vec model created!")
        print(f"  Vocabulary size: {len(w2v_model.key_to_index)}")
        print(f"  Vector dimensions: {w2v_model.vector_size}")
else:
    print("Creating simple embeddings for demonstration...")
    # Fallback: create simple embeddings
    class SimpleEmbeddings:
        def __init__(self):
            self.key_to_index = {'food':0, 'service':1, 'bad':2, 'good':3,
                                'catering':4, 'restaurant':5, 'delicious':6,
                                'terrible':7, 'amazing':8, 'great':9}
            self.vector_size = 50
            self.vectors = np.random.randn(len(self.key_to_index), self.vector_size)

        def most_similar(self, word, topn=10):
            if word not in self.key_to_index:
                return []
            idx = self.key_to_index[word]
            similarities = np.dot(self.vectors, self.vectors[idx]) / (
                np.linalg.norm(self.vectors, axis=1) * np.linalg.norm(self.vectors[idx])
            )
            most_sim = np.argsort(similarities)[::-1][1:topn+1]
            return [(list(self.key_to_index.keys())[i], similarities[i]) for i in most_sim]

        def __getitem__(self, key):
            if key in self.key_to_index:
                return self.vectors[self.key_to_index[key]]
            return None

    w2v_model = SimpleEmbeddings()
    print("✓ Simple embeddings created!")

# ============================================
# EXAMINE WORD VECTORS
# ============================================
print("\n" + "="*60)
print("EXAMINING WORD VECTORS")
print("="*60)

# Get word vectors for specific words
test_words = ['food', 'service', 'good', 'bad', 'catering']

print("\nWord Vector Properties:")
for word in test_words:
    if word in w2v_model:
        vector = w2v_model[word]
        print(f"  '{word}': vector shape = {vector.shape}, norm = {np.linalg.norm(vector):.4f}")
        print(f"    First 10 dimensions: {vector[:10]}")
    else:
        print(f"  '{word}': Not in vocabulary")

# ============================================
# FIND MOST SIMILAR WORDS
# ============================================
print("\n" + "="*60)
print("MOST SIMILAR WORDS")
print("="*60)

words_to_check = ['berlin', 'food', 'good', 'service', 'catering']

for word in words_to_check:
    print(f"\nMost similar words to '{word}':")
    try:
        similar = w2v_model.most_similar(word, topn=5)
        for sim_word, score in similar:
            print(f"  {sim_word}: {score:.4f}")
    except:
        if word in w2v_model:
            # Manual calculation for simple model
            similar = w2v_model.most_similar(word, topn=5)
            for sim_word, score in similar:
                print(f"  {sim_word}: {score:.4f}")
        else:
            print(f"  '{word}' not in vocabulary")

# ============================================
# WORD ANALOGIES
# ============================================
print("\n" + "="*60)
print("WORD ANALOGIES")
print("="*60)

try:
    # Try common analogies
    analogies = [
        ('good', 'better', 'bad'),
        ('food', 'eat', 'drink'),
        ('service', 'good', 'bad')
    ]

    for a, b, c in analogies:
        if all(w in w2v_model for w in [a, b, c]):
            result = w2v_model.most_similar(positive=[b, c], negative=[a], topn=1)
            print(f"'{a}' is to '{b}' as '{c}' is to '{result[0][0]}'")
        else:
            print(f"Analogy not possible: {a}:{b}:{c}")
except:
    print("Analogy not available for this model")

# ============================================
# VISUALIZE WORD VECTORS WITH t-SNE
# ============================================
print("\n" + "="*60)
print("VISUALIZING WORD VECTORS")
print("="*60)

# Select words to visualize
visual_words = ['food', 'service', 'bad', 'good', 'catering', 'restaurant',
                'delicious', 'terrible', 'amazing', 'great', 'rude', 'late',
                'delivery', 'experience', 'dinner', 'staff', 'customer']

# Get vectors for selected words
word_vectors = []
available_words = []

for word in visual_words:
    if word in w2v_model:
        word_vectors.append(w2v_model[word])
        available_words.append(word)

word_vectors = np.array(word_vectors)
print(f"Visualizing {len(word_vectors)} word vectors")

# ============================================
# t-SNE Visualization
# ============================================
print("\n1. t-SNE Visualization...")

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(available_words)-1))
word_vectors_tsne = tsne.fit_transform(word_vectors)

# Create DataFrame for plotting
tsne_df = pd.DataFrame({
    'x': word_vectors_tsne[:, 0],
    'y': word_vectors_tsne[:, 1],
    'word': available_words
})

# Plot using matplotlib
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(tsne_df['x'], tsne_df['y'], c=range(len(tsne_df)),
                    cmap='viridis', s=100, alpha=0.7)

# Add labels
for i, row in tsne_df.iterrows():
    ax.annotate(row['word'], (row['x'], row['y']),
                fontsize=12, fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))

plt.title('t-SNE Visualization of Word Embeddings', fontsize=16, fontweight='bold')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.grid(True, alpha=0.3)
plt.colorbar(scatter)
plt.tight_layout()
plt.show()

# ============================================
# Interactive Plotly Visualization
# ============================================
print("\n2. Interactive Plotly Visualization...")

# PCA for better visualization
pca = PCA(n_components=2)
word_vectors_pca = pca.fit_transform(word_vectors)

pca_df = pd.DataFrame({
    'x': word_vectors_pca[:, 0],
    'y': word_vectors_pca[:, 1],
    'word': available_words,
    'category': ['negative' if w in ['bad', 'terrible', 'rude', 'late']
                 else 'positive' if w in ['good', 'great', 'amazing', 'delicious']
                 else 'neutral' for w in available_words]
})

# Create interactive plot
fig = px.scatter(pca_df, x='x', y='y', text='word',
                 color='category',
                 title='Word Embeddings Visualization (PCA)',
                 labels={'x': 'PCA Component 1', 'y': 'PCA Component 2'},
                 color_discrete_map={'positive': 'green', 'negative': 'red', 'neutral': 'blue'},
                 hover_data={'word': True})

fig.update_traces(textposition='top center', marker=dict(size=20))
fig.update_layout(height=600, showlegend=True, font=dict(size=12))
fig.show()

# ============================================
# VISUALIZE WORD CLUSTERS
# ============================================
print("\n3. Word Clusters Visualization...")

# Group words by categories
categories = {
    'Positive': ['good', 'great', 'amazing', 'delicious', 'wonderful'],
    'Negative': ['bad', 'terrible', 'horrible', 'rude', 'late'],
    'Service': ['service', 'delivery', 'staff', 'customer'],
    'Food': ['food', 'restaurant', 'dinner', 'catering']
}

# Create cluster visualization
fig, ax = plt.subplots(figsize=(14, 10))
colors = ['green', 'red', 'blue', 'purple']
markers = ['o', 's', '^', 'D']

for idx, (category, words) in enumerate(categories.items()):
    cat_vectors = []
    cat_words = []
    for word in words:
        if word in w2v_model:
            cat_vectors.append(w2v_model[word])
            cat_words.append(word)

    if cat_vectors:
        cat_vectors = np.array(cat_vectors)
        # Reduce to 2D using PCA
        cat_pca = PCA(n_components=2).fit_transform(cat_vectors)

        ax.scatter(cat_pca[:, 0], cat_pca[:, 1],
                  label=category, c=colors[idx], marker=markers[idx], s=200, alpha=0.7)

        for i, word in enumerate(cat_words):
            ax.annotate(word, (cat_pca[i, 0], cat_pca[i, 1]),
                       fontsize=10, fontweight='bold')

ax.set_title('Word Embeddings by Category', fontsize=16, fontweight='bold')
ax.set_xlabel('PCA Component 1')
ax.set_ylabel('PCA Component 2')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================
# VECTOR ARITHMETIC DEMONSTRATION
# ============================================
print("\n" + "="*60)
print("VECTOR ARITHMETIC DEMONSTRATION")
print("="*60)

def vector_operation(positive, negative=[], topn=5):
    """Demonstrate vector arithmetic"""
    try:
        result = w2v_model.most_similar(positive=positive, negative=negative, topn=topn)
        return result
    except:
        return None

# Test vector arithmetic
tests = [
    (['good', 'service'], ['bad']),
    (['food', 'delicious'], ['terrible']),
    (['catering', 'good'], ['bad']),
]

for test in tests:
    if len(test) == 2:
        pos, neg = test
        print(f"\n'{pos[0]}' + '{pos[1]}' - '{neg[0]}' = ?")
        result = vector_operation(pos, neg)
        if result:
            for word, score in result[:3]:
                print(f"  {word}: {score:.4f}")

# ============================================
# GET WORD VECTORS FOR NLP PIPELINE
# ============================================
print("\n" + "="*60)
print("USING WORD VECTORS IN NLP PIPELINE")
print("="*60)

def get_word_vector(word):
    """Get vector for a word"""
    if word in w2v_model:
        return w2v_model[word]
    return None

def get_document_vector(text, method='average'):
    """Convert document to vector using word embeddings"""
    words = clean_text(text).split()
    vectors = []

    for word in words:
        vec = get_word_vector(word)
        if vec is not None:
            vectors.append(vec)

    if not vectors:
        return np.zeros(w2v_model.vector_size)

    if method == 'average':
        return np.mean(vectors, axis=0)
    elif method == 'sum':
        return np.sum(vectors, axis=0)
    else:
        return np.mean(vectors, axis=0)

def clean_text(text):
    """Simple text cleaner"""
    import re
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

# Test document vectors
sample_texts = [
    "The food was terrible",
    "Great service and amazing food",
    "Rude staff and late delivery"
]

print("\nDocument vectors for sample texts:")
for text in sample_texts:
    vec = get_document_vector(text)
    print(f"  '{text}': vector shape = {vec.shape}")
    print(f"    First 5 dimensions: {vec[:5]}")

print("\n" + "="*60)
print("✅ Word Embedding Analysis Complete!")
print("="*60)

Gensim not installed. Installing...
Run: pip install gensim
LOADING PRETRAINED WORD EMBEDDINGS
Creating simple embeddings for demonstration...
✓ Simple embeddings created!

EXAMINING WORD VECTORS

Word Vector Properties:


In [ ]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load data
try:
    df = pd.read_csv('datasets/tweets.csv')
except:
    df = pd.DataFrame({
        'text': [
            "The food was cold and terrible",
            "Great service and amazing food",
            "Rude staff and late delivery",
            "Excellent catering for the event",
            "Food poisoning from the meal",
            "Horrible catering service"
        ]
    })

# Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|@\w+|#|[^a-zA-Z\s]', '', text)
    return text

df['cleaned'] = df['text'].apply(clean_text)

# 1. Train vectorizer on tweets
vectorizer = TfidfVectorizer(min_df=1, max_df=0.9, ngram_range=(1,2))
tweet_vectors = vectorizer.fit_transform(df['cleaned'])  # Vectorize all tweets

# 2. Vectorize query
query = "bad catering service"
query_vector = vectorizer.transform([clean_text(query)])  # Vectorize query

# 3. Calculate similarity
similarities = cosine_similarity(query_vector, tweet_vectors).flatten()
df['similarity'] = similarities

# 4. Rank tweets by similarity
df_ranked = df.sort_values('similarity', ascending=False)

print("Query:", query)
print("\nAll tweets ranked by similarity:")
for idx, row in df_ranked.iterrows():
    print(f"  {row['similarity']:.4f} | {row['text']}")

# 5. Find bad catering tweets
bad_tweets = df[df['similarity'] > 0].sort_values('similarity', ascending=False)
print(f"\nFound {len(bad_tweets)} bad catering tweets:")
for idx, row in bad_tweets.iterrows():
    print(f"  {row['similarity']:.4f} | {row['text']}")